# 🧪 Lab 2: ChromaDB Setup, Data Ingestion & Retrieval Pipeline
**Vector Store Architecture & Query Execution (Slide 5)**

In this notebook, we master the production vector store lifecycle using **ChromaDB**:
1. **ChromaDB Environment Setup:** Persistent vs Ephemeral clients, telemetry configuration, and collection initialization.
2. **HNSW + Squared L2 Configuration:** Understanding the default `hnsw:space = "l2"` metric and graph parameters.
3. **Multimodal & Document Ingestion:** Batch addition of documents, IDs, and metadata.
4. **Semantic Retrieval Process:** Top-$K$ nearest neighbor search, distance conversion, and metadata filtering.


---
## Step 1: ChromaDB Setup & Client Initialization

ChromaDB provides two primary modes of operation:
- `chromadb.Client()`: In-memory ephemeral client (ideal for ephemeral tests and notebooks).
- `chromadb.PersistentClient(path)`: Persists vector embeddings and HNSW graph indexes to disk via SQLite and `hnswlib` files.


In [ ]:
import os
import shutil
import numpy as np
import chromadb
from chromadb.config import Settings

print(f"✔ ChromaDB Library Version: {chromadb.__version__}")

# Initialize an in-memory client for instant experimentation
client = chromadb.Client(Settings(anonymized_telemetry=False, is_persistent=False))

print("✔ ChromaDB Client successfully initialized!")


---
## Step 2: Collection Creation with HNSW + Squared L2

In ChromaDB, a **Collection** holds vector embeddings, document text, and metadata.
By default, ChromaDB configures collections with:
- `hnsw:space`: `"l2"` (Squared Euclidean distance)
- `hnsw:construction_ef`: `100` (Index construction beam width)
- `hnsw:M`: `16` (Max bi-directional links per node)
- `hnsw:search_ef`: `10` (Search beam width)


In [ ]:
# Reset existing collection if it exists
collection_name = "multimodal_knowledge"
try:
    client.delete_collection(name=collection_name)
except Exception:
    pass

# Create the collection with explicit HNSW + Squared L2 configuration
from chromadb.utils import embedding_functions
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="sentence-transformers/all-MiniLM-L6-v2")
collection = client.create_collection(
    embedding_function=embedding_fn,
    name=collection_name,
    metadata={"hnsw:space": "l2"} # Default squared Euclidean distance
)

print(f"✔ Collection '{collection.name}' created!")
print(f"✔ Collection Configuration Metadata: {collection.metadata}")


---
## Step 3: Ingestion of Multimodal & Corporate Policy Data

Let's ingest our Slide-1 items along with corporate policy passages.
ChromaDB accepts:
- `documents`: Raw text strings (automatically embedded via the default `all-MiniLM-L6-v2` embedding function)
- `metadatas`: Key-value dictionaries for attribute filtering
- `ids`: Unique string identifiers


In [ ]:
# Prepare data payload matching Slide 1 and corporate policy corpus
documents = [
    "Cat",
    "Dog",
    "On the table cat is there.",
    "a high-resolution photograph of a cute orange tabby cat sitting indoors",
    "a youtube tutorial video demonstrating how to care for domestic pet cats",
    "Remote Work & Telecommuting Guidelines: Full-time employees are eligible to telecommute and perform professional duties from home up to three days per calendar week upon manager approval.",
    "Information Security Protocol: All employee laptops must utilize full-disk AES-256 encryption and connect through the corporate corporate VPN when accessing internal infrastructure.",
    "Paid Time Off (PTO) Policy: Salaried employees accrue 1.75 days of paid time off per calendar month worked, up to a maximum rollover cap of 25 accrued days per calendar year."
]

metadatas = [
    {"source": "sketch", "category": "pets", "modality": "text"},
    {"source": "sketch", "category": "pets", "modality": "text"},
    {"source": "sketch", "category": "pets", "modality": "text"},
    {"source": "sketch", "category": "pets", "modality": "image"},
    {"source": "sketch", "category": "pets", "modality": "video"},
    {"source": "policy", "category": "hr", "modality": "text"},
    {"source": "policy", "category": "infosec", "modality": "text"},
    {"source": "policy", "category": "hr", "modality": "text"}
]

ids = [f"doc_{i+1:03d}" for i in range(len(documents))]

# Ingest into ChromaDB
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

total_count = collection.count()
print("=" * 65)
print(f"✔ Ingestion Complete: {total_count} documents stored in ChromaDB index.")
print("=" * 65)


---
## Step 4: Semantic Retrieval Process & Query Execution

When we call `collection.query()`, ChromaDB performs the following steps:
1. Encodes the query text using the same embedding function.
2. Traverses the HNSW graph using Squared $L_2$ distance.
3. Returns the Top-$K$ items matching the query vector.


In [ ]:
# Query 1: Querying for our cat on the table
query_text = "Where is the cat sitting?"
results = collection.query(
    query_texts=[query_text],
    n_results=3
)

print(f"QUERY: '{query_text}'")
print("=" * 70)
print(f"{'Rank':4s} | {'ID':8s} | {'Squared L2':10s} | {'Sim Score':10s} | {'Document Text'}")
print("-" * 70)

for rank in range(len(results['ids'][0])):
    doc_id = results['ids'][0][rank]
    doc_text = results['documents'][0][rank]
    l2_dist = results['distances'][0][rank]
    # Convert ChromaDB Squared L2 distance to normalized similarity: Sim = 1 - d² / 2
    sim_score = max(0.0, 1.0 - (l2_dist / 2.0))
    print(f"#{rank+1:<3d} | {doc_id:8s} | {l2_dist:10.4f} | {sim_score:10.4f} | {doc_text[:40]}...")
print("=" * 70)


### Metadata Filtering with the `where` Clause
ChromaDB supports pre-filtering results using metadata attributes:


In [ ]:
# Query 2: Querying for corporate remote work with metadata filtering
query_hr = "working from home telecommuting schedule"
filtered_results = collection.query(
    query_texts=[query_hr],
    where={"category": "hr"},
    n_results=2
)

print(f"HR FILTERED QUERY: '{query_hr}' (where category == 'hr')")
print("=" * 70)
for rank in range(len(filtered_results['ids'][0])):
    d_id = filtered_results['ids'][0][rank]
    d_text = filtered_results['documents'][0][rank]
    d_dist = filtered_results['distances'][0][rank]
    d_meta = filtered_results['metadatas'][0][rank]
    sim = max(0.0, 1.0 - (d_dist / 2.0))
    print(f"Rank #{rank+1} (ID: {d_id}, L2 Distance: {d_dist:.4f}, Sim: {sim:.4f}):")
    print(f"  Category: {d_meta['category']} | Source: {d_meta['source']}")
    print(f"  Passage : '{d_text}'")
    print("-" * 70)


---
## Summary
- **ChromaDB Setup:** Ephemeral or persistent clients manage collections with HNSW vector graphs.
- **Default Space:** `hnsw:space = "l2"` avoids expensive square root operations while strictly preserving nearest neighbor ranking.
- **Conversion:** Distance $d^2$ converts to normalized similarity via $	ext{Sim} = 1 - rac{d^2}{2}$.
- **Filtering:** Metadata `where` queries prune the search space before graph traversal.
